In [1]:
import polars as pl
from pathlib import Path
import datetime

COLUMN_NAMES = ["amount", "provider", "rate", "date", "compte", "owner","type"]

MONTH_MAP = {
    "JAN": "01", "FEB": "02", "MAR": "03", "APR": "04",
    "MAY": "05", "JUN": "06", "JUL": "07", "AUG": "08",
    "SEP": "09", "OCT": "10", "NOV": "11", "DEC": "12"
}


In [2]:

def parse_date(val: str) -> datetime.date:
    val = val.strip()
    day = val[:2]
    month = MONTH_MAP[val[2:5].upper()]
    year = "20" + val[5:]
    return datetime.date.fromisoformat(f"{year}-{month}-{day}")

In [3]:
def load_csv(path: str | Path) -> pl.DataFrame:
    df = pl.read_csv(
        path,
        has_header=False,
        new_columns=COLUMN_NAMES,
        infer_schema=False,
    )

    df = df.with_columns(pl.all().str.strip_chars())


    df = df.with_columns([
        pl.col("amount").str.strip_chars().cast(pl.Float64),
        pl.col("rate")
          .str.strip_chars()
          .str.replace("%", "")
          .str.strip_chars()
          .cast(pl.Float64)
          .truediv(100)
          .round(10),

    ])

    df = df.with_columns(
        pl.col("date").map_elements(parse_date, return_dtype=pl.Date)
    )
    df = df.sort("date")


    return df

In [4]:
input_path = "data/gic_bronze.csv"
df = load_csv(input_path)


InvalidOperationError: conversion from `str` to `f64` failed in column 'amount' for 1 out of 19 values: ["amount"]

This error occurred in the following expression:
	col("amount").str.strip_chars([null]).strict_cast(Float64)


In [150]:
df.write_csv("data/memory_gic.csv")

In [151]:
def summarize_issue(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.group_by(["provider", "owner"])
          .agg(pl.col("amount").sum().alias("total_amount"))
          .with_columns(
              (pl.col("total_amount")>100000).alias("Above_100K")
          )
          .sort("total_amount", descending=True)
    )

In [152]:
summarize_issue(df).write_csv("data/lookup_gic.csv")


In [153]:
def summarize_timing(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.with_columns(
            pl.col("date").dt.strftime("%Y-%m").alias("year_month")
        )
        .group_by(["year_month","type"])
        .agg(pl.col("amount").sum().alias("total_amount"))
        .sort(["year_month","type"])
    )



In [154]:
summarize_timing(df).write_csv("data/timing_gic.csv")

In [167]:
TYPE_ORDER = ["TOTAL", "NR", "RRSP"]

def compute_summary(df: pl.DataFrame) -> pl.DataFrame:

    summary = (
        df.with_columns(
            (pl.col("amount") * pl.col("rate")).alias("interest")
        )
        .group_by("type")
        .agg(
            pl.col("amount").sum().alias("sum_total"),
            pl.col("interest").sum().alias("sum_interest")
        )
        .pipe(lambda df: pl.concat([
            pl.DataFrame({
                "type": ["TOTAL"],
                "sum_total": [df["sum_total"].sum()],
                "sum_interest": [df["sum_interest"].sum()]
            }),
            df
        ]))
        .with_columns(pl.col("type").cast(pl.Enum(TYPE_ORDER)))
        .sort("type")
    )

    return summary

In [168]:
compute_summary(df)

type,sum_total,sum_interest
enum,f64,f64
"""TOTAL""",927510.0,34488.8905
"""NR""",570510.0,21408.7405
"""RRSP""",357000.0,13080.15


In [169]:
compute_summary(df.filter(pl.col("owner")=="Sandrine"))

type,sum_total,sum_interest
enum,f64,f64
"""TOTAL""",640005.0,23890.468
"""NR""",366005.0,13855.068
"""RRSP""",274000.0,10035.4


In [160]:
result = (
    df.with_columns(
        (pl.col("amount") * pl.col("rate")).alias("interest")
    )
    .group_by(["type"])
    .agg([
        pl.col("amount").sum().alias("sum_total"),
        pl.col("interest").sum().alias("sum_interest")
    ])
)

total_row = pl.DataFrame({
    "type": ["TOTAL"],
    "sum_total": [result["sum_total"].sum()],
    "sum_interest": [result["sum_interest"].sum()]
})

result = pl.concat([result, total_row])

In [161]:
result

type,sum_total,sum_interest
str,f64,f64
"""RRSP""",357000.0,13080.15
"""NR""",570510.0,21408.7405
"""TOTAL""",927510.0,34488.8905


In [163]:
result = (
    df.with_columns(
        (pl.col("amount") * pl.col("rate")).alias("interest")
    )
    .filter(pl.col("owner").is_in(["Sandrine"]))
    .group_by(["type"])
    .agg([
        pl.col("amount").sum().alias("sum_total"),
        pl.col("interest").sum().alias("sum_interest")
    ])
)

total_row = pl.DataFrame({
    "type": ["TOTAL"],
    "sum_total": [result["sum_total"].sum()],
    "sum_interest": [result["sum_interest"].sum()]
})

result = pl.concat([result, total_row])
result

type,sum_total,sum_interest
str,f64,f64
"""NR""",366005.0,13855.068
"""RRSP""",274000.0,10035.4
"""TOTAL""",640005.0,23890.468


In [127]:
df.group_by(["provider","owner"]).agg(pl.col("amount").sum().alias("total_amount"))


provider,owner,total_amount
str,str,f64
"""PEOPLE TRUST GIC A""","""Sandrine""",78000.0
"""HOMEQTY BK GIC AN""","""Sandrine""",82000.0
"""HOME Bank""","""Serge""",29900.0
"""FAIRSTONE BK GIC A""","""Sandrine""",144000.0
"""VERSA BK GIC AN""","""Sandrine""",44005.0
…,…,…
"""B2B BK GIC AN""","""Sandrine""",25000.0
"""CONCENTRA BK GIC""","""Sandrine""",50000.0
"""ENBRIDG HYB-2027""","""Serge""",3000.0


In [ ]:







def run_pipeline(input_path: str | Path, output_csv: str | Path) -> pl.DataFrame:
    df = load_csv(input_path)

    df = df.sort("date")

    # Distinct values
    print("=== Distinct providers ===")
    print(df["provider"].unique().to_list())
    print("=== Distinct comptes ===")
    print(df["compte"].unique().to_list())

    # Summary
    summary = summarize(df)
    print("=== Amount by provider & compte ===")
    print(summary)

    # Output
    df.write_csv(output_csv)
    print(f"\nCSV saved → {output_csv}")

    return df


if __name__ == "__main__":
    df = run_pipeline("input.csv", "output.csv")